# Data Integration

In [58]:
import pandas as pd
import recordlinkage as rl

In [59]:
df_athletics = pd.read_csv("cleaned_athletics_data.csv")
df_admissions = pd.read_csv("2023-24_cleaned_admissions_data.csv")

In [60]:
df_athletics.head()

,unitid,opeid,institution_name,city,state,zip,classification_name,institution_type,institution_years,EFTotalCount,STUDENTAID_TOTAL,RECRUITEXP_TOTAL,GRND_TOTAL_REVENUE,GRND_TOTAL_EXPENSE
0,100654,100200,alabama a & m university,normal,AL,35762,NCAA Division I-FCS,public,4-year or above,5219,3654220,114085,12639752,12639752
1,100663,105200,university of alabama at birmingham,birmingham,AL,35294-0110,NCAA Division I-FBS,public,4-year or above,9805,8176953,789941,43271595,42655846
2,100706,105500,university of alabama in huntsville,huntsville,AL,35899,NCAA Division II without football,public,4-year or above,5642,2597502,58361,8054352,8054352
3,100724,100500,alabama state university,montgomery,AL,36104-0271,NCAA Division I-FCS,public,4-year or above,2959,4602765,214415,14050365,14050365
4,100751,105100,the university of alabama,tuscaloosa,AL,35487-0100,NCAA Division I-FBS,public,4-year or above,29956,15912938,7096598,243096720,243096720


In [57]:
df_admissions.head()

,Unnamed: 0,unitid,opeid,institution_name,city,state,zip,main_campus,num_campuses,institution_type,...,ACT_median25,ACT_median75,num_undergrad,avg_net_price_public,avg_net_price_private,in_state_tuition,out_state_tuition,prop_low_income,institution_years,test_score_req
0,0,100654,100200.0,alabama a & m university,normal,AL,35762,1,1,public,...,14.0,19.0,5726.0,14559.0,NaN,10024.0,18634.0,0.6441,4-year,Considered but not required
1,1,100663,105200.0,university of alabama at birmingham,birmingham,AL,35294-0110,1,1,public,...,22.0,30.0,12118.0,17727.0,NaN,8832.0,21864.0,0.3318,4-year,Considered but not required
2,2,100690,2503400.0,amridge university,montgomery,AL,36117-3553,1,1,private nonprofit,...,NaN,NaN,226.0,NaN,NaN,NaN,NaN,0.6842,4-year,NaN
3,3,100706,105500.0,university of alabama in huntsville,huntsville,AL,35899,1,1,public,...,25.0,31.0,6650.0,19880.0,NaN,11770.0,24662.0,0.2250,4-year,Considered but not required
4,4,100724,100500.0,alabama state university,montgomery,AL,36104-0271,1,1,public,...,16.0,20.0,3322.0,13889.0,NaN,11248.0,19576.0,0.7203,4-year,Considered but not required


In [100]:
df_athletics.shape, df_admissions.shape

((2040, 14), (6430, 25))

In [103]:
len(df_athletics.unitid.unique()), len(df_admissions.unitid.unique())

(2040, 6430)

# Record Linkage

In [93]:
indexer = rl.Index()
indexer.block("state")
pairs = indexer.index(df_admissions, df_athletics)

compare = rl.Compare()
compare.string("institution_name", "institution_name", method = "levenshtein", threshold = 0.85, label = "matched")
features = compare.compute(pairs, df_admissions, df_athletics)

candidates = features[features.sum(axis=1) == 1]
candidates

,,matched
0,0,1.0
1,1,1.0
3,2,1.0
4,3,1.0
5,4,1.0
...,...,...
3573,1959,1.0
3574,1945,1.0
3575,1946,1.0
5682,1953,1.0


In [128]:
linked_list = []
for adm_idx, ath_idx in candidates.index:
    ath_row = df_athletics.loc[ath_idx]
    adm_row = df_admissions.loc[adm_idx]

    linked_list.append({
        "unitid": ath_row["unitid"],
        "opeid": ath_row["opeid"],
        "institution_name": ath_row["institution_name"],
        "city": ath_row["city"],
        "state": ath_row["state"],
        "zip": ath_row["zip"],
        "classification_name": ath_row["classification_name"],
        "institution_type": adm_row["institution_type"],
        "institution_years": adm_row["institution_years"],
        "admission_rate": adm_row["admission_rate"],
        "SAT_reading25": adm_row["SAT_reading25"],
        "SAT_reading75": adm_row["SAT_reading75"],
        "SAT_math25": adm_row["SAT_math25"],
        "SAT_math75": adm_row["SAT_math75"],
        "ACT_median25": adm_row["ACT_median25"],
        "ACT_median75": adm_row["ACT_median75"],
        "num_undergrad": adm_row["num_undergrad"],
        "avg_net_price_public": adm_row["avg_net_price_public"],
        "avg_net_price_private": adm_row["avg_net_price_private"],
        "in_state_tuition": adm_row["in_state_tuition"],
        "out_state_tuition": adm_row["out_state_tuition"],
        "prop_low_income": adm_row["prop_low_income"],
        "test_score_req": adm_row["test_score_req"],
        "EFTotalCount": ath_row["EFTotalCount"],
        "STUDENTAID_TOTAL": ath_row["STUDENTAID_TOTAL"],
        "RECRUITEXP_TOTAL": ath_row["RECRUITEXP_TOTAL"],
        "GRND_TOTAL_REVENUE": ath_row["GRND_TOTAL_REVENUE"],
        "GRND_TOTAL_EXPENSE": ath_row["GRND_TOTAL_EXPENSE"]
    })

linked_df = pd.DataFrame(linked_list)

In [129]:
linked_df

,unitid,opeid,institution_name,city,state,zip,classification_name,institution_type,institution_years,admission_rate,...,avg_net_price_private,in_state_tuition,out_state_tuition,prop_low_income,test_score_req,EFTotalCount,STUDENTAID_TOTAL,RECRUITEXP_TOTAL,GRND_TOTAL_REVENUE,GRND_TOTAL_EXPENSE
0,100654,100200,alabama a & m university,normal,AL,35762,NCAA Division I-FCS,public,4-year,0.6622,...,NaN,10024.0,18634.0,0.6441,Considered but not required,5219,3654220,114085,12639752,12639752
1,100663,105200,university of alabama at birmingham,birmingham,AL,35294-0110,NCAA Division I-FBS,public,4-year,0.8842,...,NaN,8832.0,21864.0,0.3318,Considered but not required,9805,8176953,789941,43271595,42655846
2,100706,105500,university of alabama in huntsville,huntsville,AL,35899,NCAA Division II without football,public,4-year,0.7425,...,NaN,11770.0,24662.0,0.2250,Considered but not required,5642,2597502,58361,8054352,8054352
3,100724,100500,alabama state university,montgomery,AL,36104-0271,NCAA Division I-FCS,public,4-year,0.9564,...,NaN,11248.0,19576.0,0.7203,Considered but not required,2959,4602765,214415,14050365,14050365
4,100751,105100,the university of alabama,tuscaloosa,AL,35487-0100,NCAA Division I-FBS,public,4-year,0.7582,...,NaN,11900.0,33200.0,0.1799,Considered but not required,29956,15912938,7096598,243096720,243096720
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260,243577,2100000,universidad politecnica de puerto rico,hato rey,PR,00919,Other,private nonprofit,4-year,NaN,...,16338.0,9810.0,9810.0,0.6232,NaN,1795,215000,0,510359,510359
2261,241410,393600,pontifical catholic university of puerto rico-...,ponce,PR,00717-9997,Other,private nonprofit,4-year,0.9730,...,8264.0,6238.0,6238.0,0.8346,Considered but not required,3357,604646,0,936090,936090
2262,241739,2587500,universidad ana g. mendez-cupey campus,san juan,PR,00926-2602,Other,private nonprofit,4-year,NaN,...,7668.0,6920.0,6920.0,0.8266,NaN,3226,1462638,2000,2061404,2061404
2263,243179,394300,university of puerto rico-humacao,humacao,PR,00792,Other,private nonprofit,4-year,0.8488,...,8618.0,7050.0,14125.0,0.6214,Neither required nor recommended,2325,0,0,167556,167556


In [130]:
linked_df.drop_duplicates(["institution_name", "unitid"], inplace=True, ignore_index=True)

In [131]:
linked_df

,unitid,opeid,institution_name,city,state,zip,classification_name,institution_type,institution_years,admission_rate,...,avg_net_price_private,in_state_tuition,out_state_tuition,prop_low_income,test_score_req,EFTotalCount,STUDENTAID_TOTAL,RECRUITEXP_TOTAL,GRND_TOTAL_REVENUE,GRND_TOTAL_EXPENSE
0,100654,100200,alabama a & m university,normal,AL,35762,NCAA Division I-FCS,public,4-year,0.6622,...,NaN,10024.0,18634.0,0.6441,Considered but not required,5219,3654220,114085,12639752,12639752
1,100663,105200,university of alabama at birmingham,birmingham,AL,35294-0110,NCAA Division I-FBS,public,4-year,0.8842,...,NaN,8832.0,21864.0,0.3318,Considered but not required,9805,8176953,789941,43271595,42655846
2,100706,105500,university of alabama in huntsville,huntsville,AL,35899,NCAA Division II without football,public,4-year,0.7425,...,NaN,11770.0,24662.0,0.2250,Considered but not required,5642,2597502,58361,8054352,8054352
3,100724,100500,alabama state university,montgomery,AL,36104-0271,NCAA Division I-FCS,public,4-year,0.9564,...,NaN,11248.0,19576.0,0.7203,Considered but not required,2959,4602765,214415,14050365,14050365
4,100751,105100,the university of alabama,tuscaloosa,AL,35487-0100,NCAA Division I-FBS,public,4-year,0.7582,...,NaN,11900.0,33200.0,0.1799,Considered but not required,29956,15912938,7096598,243096720,243096720
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015,243188,1092200,university of puerto rico-utuado,utuado,PR,00641,Other,public,4-year,0.6411,...,NaN,5354.0,5354.0,0.8150,Neither required nor recommended,284,40800,1400,231359,231359
2016,243221,710800,university of puerto rico-rio piedras,san juan,PR,00931-0000,NCAA Division II without football,public,4-year,0.5320,...,NaN,5324.0,5324.0,0.6475,Neither required nor recommended,7768,652000,0,1523196,1523196
2017,243443,393700,universidad del sagrado corazon,santurce,PR,00907,Other,private nonprofit,4-year,0.5341,...,13024.0,6360.0,6360.0,0.7081,Neither required nor recommended,3468,537672,0,923710,923710
2018,243577,2100000,universidad politecnica de puerto rico,hato rey,PR,00919,Other,private nonprofit,4-year,NaN,...,16338.0,9810.0,9810.0,0.6232,NaN,1795,215000,0,510359,510359


In [132]:
linked_df.isna().sum()

unitid                      0
opeid                       0
institution_name            0
city                        0
state                       0
zip                         0
classification_name         0
institution_type            0
institution_years           0
admission_rate            733
SAT_reading25            1137
SAT_reading75            1137
SAT_math25               1137
SAT_math75               1137
ACT_median25             1166
ACT_median75             1166
num_undergrad               2
avg_net_price_public      864
avg_net_price_private    1159
in_state_tuition            2
out_state_tuition           2
prop_low_income             2
test_score_req            733
EFTotalCount                0
STUDENTAID_TOTAL            0
RECRUITEXP_TOTAL            0
GRND_TOTAL_REVENUE          0
GRND_TOTAL_EXPENSE          0
dtype: int64

In [133]:
linked_df.to_csv("integrated_dataset.csv", index=False)